## Base Model training

We start by training and testing VGG16 and Densenet121 on the task. <br>
We then go on to evaluate the performance MMSEN on the task.

In [14]:
import sys
from google.colab import drive
drive.mount("/content/drive")
PROJECT_ROOT = "/content/drive/MyDrive/Projects/MMSEN"
sys.path.insert(0, PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!kaggle datasets download -d andrewmvd/metastatic-tissue-classification-patchcamelyon

Dataset URL: https://www.kaggle.com/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon
License(s): CC0-1.0
Resuming from 2276458496 bytes (5742867652 bytes left)...
100% 7.47G/7.47G [05:17<00:00, 18.1MB/s] 



In [3]:
!unzip metastatic-tissue-classification-patchcamelyon.zip

Archive:  metastatic-tissue-classification-patchcamelyon.zip
  inflating: Labels/Labels/camelyonpatch_level_2_split_test_y.h5  
  inflating: Labels/Labels/camelyonpatch_level_2_split_train_y.h5  
  inflating: Labels/Labels/camelyonpatch_level_2_split_valid_y.h5  
  inflating: Metadata/Metadata/test_metadata.csv  
  inflating: Metadata/Metadata/train_metadata.csv  
  inflating: Metadata/Metadata/valid_metadata.csv  
  inflating: camelyonpatch_level_2_split_train_mask/camelyonpatch_level_2_split_train_mask.h5  
  inflating: pcam/test_split.h5      
  inflating: pcam/training_split.h5  
  inflating: pcam/validation_split.h5  


In [15]:
from torchvision import models, transforms
import torch
from torch import nn

from src.train import train_loop
from src.evaluation import plot_metrics, confusion_matrix, vis_results
from src.utils import assemble_base_mmsen

ModuleNotFoundError: No module named 'src.utils'

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

In [ ]:
# define a transform, now normalize using the computed mean and std,
# also add random horizontal and vertical flipping for dataset augmentation

mean = [0.7008, 0.5384, 0.6916]
std = [0.2350, 0.2774, 0.2129]

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

### VGG16

In [ ]:
# Load VGG16 with pretrained ImageNet weights
vgg16 = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

In [ ]:
vgg16.classifier

In [ ]:
class VGG16_GAP_Classifier(nn.Module):
  
  def __init__(self):
    super().__init__()
    self.features = vgg16.features
    self.classifier = nn.Linear(512, 1)

  def forward(self, x):
    x = self.features(x)
    x = torch.mean(x, dim=(2, 3))
    x = self.classifier(x)
    return x

In [ ]:
vgg16 = VGG16_GAP_Classifier()

In [ ]:
### pass the model to the correct device
vgg16 = vgg16.to(device, memory_format=torch.channels_last)

### freeze all layers apart from the classifier

for param in vgg16.parameters():
  param.requires_grad = False

for param in vgg16.classifier.parameters():
  param.requires_grad = True

In [ ]:
vgg16, train_loss, test_loss, roc_auc, precision, sensitivity, specificity, f1_score, balanced_accuracy, mcc, classes_per_epoch = train_loop(
    vgg16, 
    "MMSEN/models/VGG/vgg16_classifier_10epochs",
    train_transform,
    test_transform
)

In [ ]:
plot_metrics(train_loss, test_loss, roc_auc, precision, sensitivity, specificity, f1_score, balanced_accuracy, mcc)

In [ ]:
confusion_matrix(classes_per_epoch[-1])

In [ ]:
vis_results(vgg16)

### DenseNet121

In [ ]:
# Load DenseNet121 with pretrained ImageNet weights
densenet121 = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)

In [ ]:
# DenseNet121 already uses GAP for the classification layer
densenet121.classifier

In [ ]:
densenet121.classifier = nn.Linear(in_features=1024, out_features=1)

In [ ]:
### pass the model to the correct device
densenet121 = densenet121.to(device, memory_format=torch.channels_last)

### freeze all layers apart from the classifier

for param in densenet121.parameters():
  param.requires_grad = False

for param in densenet121.classifier.parameters():
  param.requires_grad = True

In [ ]:
densenet121, train_loss, test_loss, roc_auc, precision, sensitivity, specificity, f1_score, balanced_accuracy, mcc, classes_per_epoch = train_loop(densenet121, "models/DenseNet121/densenet121_classifier_10epochs")

In [ ]:
plot_metrics(train_loss, test_loss, roc_auc, precision, sensitivity, specificity, f1_score, balanced_accuracy, mcc)

In [ ]:
confusion_matrix(classes_per_epoch[-1])

In [ ]:
vis_results(densenet121)

### MMSEN

In [ ]:
### pass the model to the correct device
mmsen = assemble_base_mmsen()
mmsen = mmsen.to(device, memory_format=torch.channels_last)

In [ ]:
mmsen, train_loss, test_loss, roc_auc, precision, sensitivity, specificity, f1_score, balanced_accuracy, mcc, classes_per_epoch = train_loop(mmsen, "mmsen_classifier_10epochs", custom_lr=True)

In [ ]:
plot_metrics(train_loss, test_loss, roc_auc, precision, sensitivity, specificity, f1_score, balanced_accuracy, mcc)

In [ ]:
confusion_matrix(classes_per_epoch[-1])

In [ ]:
vis_results(mmsen)